# Gesture Recognition TCN - Training Notebook

This notebook trains a 1D-CNN (TCN) model for gesture recognition and exports it to ONNX format for Android deployment.

**Dataset**: Located at `../dataset/processed/`

**Classes** (5 classes):
- `swipe_up` - 上滑
- `swipe_down` - 下滑  
- `grab` - 抓
- `release` - 放
- `noise` - 噪音（其他手势/无手势）

## 1. Environment Setup

In [1]:
# Check GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA available: True
GPU: NVIDIA GeForce RTX 5070 Ti Laptop GPU


In [ ]:
# Install dependencies (uncomment if needed)
# !pip install -q mediapipe onnx onnxruntime scikit-learn tqdm

In [2]:
import os
import json
import random
import logging
from pathlib import Path
from collections import defaultdict
from copy import deepcopy

import cv2
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.utils.prune as prune
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import classification_report, confusion_matrix

# Notebook-friendly logging (print instead of logging module)
from datetime import datetime


def log_info(msg):
    """Print INFO message with timestamp - works in Jupyter notebooks."""
    ts = datetime.now().strftime("%H:%M:%S")
    print(f"[{ts}] INFO - {msg}")


def log_warn(msg):
    """Print WARNING message with timestamp - works in Jupyter notebooks."""
    ts = datetime.now().strftime("%H:%M:%S")
    print(f"[{ts}] WARNING - {msg}")


def log_err(msg):
    """Print ERROR message with timestamp - works in Jupyter notebooks."""
    ts = datetime.now().strftime("%H:%M:%S")
    print(f"[{ts}] ERROR - {msg}")

## 2. Dataset Path Configuration

In [4]:
# Dataset path - relative to this notebook
from pathlib import Path

# Get the directory where this notebook is located
NOTEBOOK_DIR = Path(__file__).parent if '__file__' in dir() else Path.cwd()
DATASET_PATH = (NOTEBOOK_DIR / "../dataset/processed").resolve()
TRAIN_DIR = DATASET_PATH / "Train"
TEST_DIR = DATASET_PATH / "Test"

# Check if dataset exists
import os

if not DATASET_PATH.exists():
    print("=" * 70)
    print("⚠️  DATASET NOT FOUND!")
    print("=" * 70)
    print(f"\n📂 Expected dataset location: {DATASET_PATH}")
    print("\n📋 Please run preprocess_videos.py first to generate the dataset:")
    print("   cd ../dataset")
    print("   python preprocess_videos.py --execute --augment")
    print("\n📋 Expected folder structure:")
    print(f"      {DATASET_PATH}/")
    print(f"      ├── Train/")
    print("      │   ├── grab/")
    print("      │   ├── swipe_up/")
    print("      │   ├── swipe_down/")
    print("      │   ├── release/")
    print("      │   └── noise/")
    print(f"      └── Test/")
    print("          ├── grab/")
    print("          ├── swipe_up/")
    print("          ├── swipe_down/")
    print("          ├── release/")
    print("          └── noise/")
    print("=" * 70)
    raise FileNotFoundError(f"Dataset not found at {DATASET_PATH}")

print("✅ Dataset found!")
print(f"   Dataset path: {DATASET_PATH}")
print(f"   Train dir: {TRAIN_DIR}")
print(f"   Test dir: {TEST_DIR}")

✅ Dataset found!
   Dataset path: D:\Scarleto\Documents\Projet\comp5201\Projet\AirGesture-xuranus\dataset\processed
   Train dir: D:\Scarleto\Documents\Projet\comp5201\Projet\AirGesture-xuranus\dataset\processed\Train
   Test dir: D:\Scarleto\Documents\Projet\comp5201\Projet\AirGesture-xuranus\dataset\processed\Test


In [5]:
# Verify dataset structure
import os


def check_dataset_structure():
    """Check if dataset has correct structure and report issues."""
    issues = []

    # Check train directory
    if not TRAIN_DIR.exists():
        issues.append(f"❌ Train directory not found: {TRAIN_DIR}")
    else:
        print(f"✅ Train directory exists: {TRAIN_DIR}")

    # Check test directory
    if not TEST_DIR.exists():
        issues.append(f"❌ Test directory not found: {TEST_DIR}")
    else:
        print(f"✅ Test directory exists: {TEST_DIR}")

    # Expected class names (5 classes)
    expected_classes = [
        "swipe_up",
        "swipe_down",
        "grab",
        "release",
        "noise",
    ]

    # Check train classes
    if TRAIN_DIR.exists():
        train_classes = [
            d for d in os.listdir(TRAIN_DIR)
            if os.path.isdir(TRAIN_DIR / d)
        ]
        missing_train = set(expected_classes) - set(train_classes)
        if missing_train:
            issues.append(f"⚠️  Missing classes in train: {missing_train}")
        print(f"   Train classes found: {train_classes}")

    # Check test classes
    if TEST_DIR.exists():
        test_classes = [
            d for d in os.listdir(TEST_DIR)
            if os.path.isdir(TEST_DIR / d)
        ]
        missing_test = set(expected_classes) - set(test_classes)
        if missing_test:
            issues.append(f"⚠️  Missing classes in test: {missing_test}")
        print(f"   Test classes found: {test_classes}")

    return issues


issues = check_dataset_structure()

if issues:
    print("\n" + "=" * 70)
    print("⚠️  DATASET STRUCTURE ISSUES DETECTED:")
    print("=" * 70)
    for issue in issues:
        print(f"   {issue}")
    print("\nPlease fix the above issues and re-run this cell.")
    print("=" * 70)
else:
    print("\n✅ Dataset structure is correct!")

✅ Train directory exists: D:\Scarleto\Documents\Projet\comp5201\Projet\AirGesture-xuranus\dataset\processed\Train
✅ Test directory exists: D:\Scarleto\Documents\Projet\comp5201\Projet\AirGesture-xuranus\dataset\processed\Test
   Train classes found: ['grab', 'noise', 'release', 'swipe_down', 'swipe_up']
   Test classes found: ['grab', 'noise', 'release', 'swipe_down', 'swipe_up']

✅ Dataset structure is correct!


## 3. Constants and Configuration

In [6]:
# Constants
SEQ_LEN = 30
NUM_LANDMARKS = 21
NUM_COORDS = 3
RAW_DIM = NUM_LANDMARKS * NUM_COORDS
NUM_CLASSES = 5  # swipe_up, swipe_down, grab, release, noise
BATCH_SIZE = 32
EPOCHS = 300
LR = 2e-3
WEIGHT_DECAY = 1e-3
PRUNE_AMOUNT = 0.20
MIN_VALID_RATIO = 0.10
PATIENCE = 40
CACHE_VERSION = "v7_5class"  # Updated version for 5 classes

# Feature computation constants
WRIST_IDX = 0
MID_FINGER_IDX = 9
FINGERTIP_IDS = [4, 8, 12, 16, 20]
BASE_IDS = [2, 5, 9, 13, 17]
PAIRS = [
    (4, 8),
    (8, 12),
    (12, 16),
    (16, 20),
    (4, 12),
    (4, 16),
    (4, 20),
    (8, 16),
    (8, 20),
    (12, 20),
]
N_PAIRS = len(PAIRS)
FINGER_CHAINS = [
    [0, 1, 2, 3, 4],
    [0, 5, 6, 7, 8],
    [0, 9, 10, 11, 12],
    [0, 13, 14, 15, 16],
    [0, 17, 18, 19, 20],
]
N_FINGERS = 5
FEATURE_DIM = RAW_DIM + RAW_DIM + 3 + N_PAIRS + N_FINGERS

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Class names (5 classes)
CLASS_NAMES = [
    "swipe_up",    # 上滑
    "swipe_down",  # 下滑
    "grab",        # 抓
    "release",     # 放
    "noise",       # 噪音
]
CLASS_TO_IDX = {n: i for i, n in enumerate(CLASS_NAMES)}

# Swipe classes (for data augmentation - no left/right swap needed)
SWIPE_CLASSES = {"swipe_up", "swipe_down"}

print(f"Device: {DEVICE}")
print(f"Feature dim: {FEATURE_DIM}")
print(f"Num classes: {NUM_CLASSES}")
print(f"Classes: {CLASS_NAMES}")

Device: cuda
Feature dim: 144
Num classes: 5
Classes: ['swipe_up', 'swipe_down', 'grab', 'release', 'noise']


## 4. Hand Landmark Detection (MediaPipe)

In [7]:
# Download MediaPipe hand landmark model
HAND_LANDMARKER_MODEL = "hand_landmarker.task"
MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task"
)


def ensure_model_file(path=HAND_LANDMARKER_MODEL):
    if os.path.exists(path):
        return path
    log_info(f"Downloading {path} ...")
    import requests

    r = requests.get(MODEL_URL, stream=True, timeout=120)
    r.raise_for_status()
    with open(path, "wb") as f:
        for chunk in r.iter_content(8192):
            f.write(chunk)
    return path


ensure_model_file()

[18:57:54] INFO - Downloading hand_landmarker.task ...


'hand_landmarker.task'

In [8]:
class HandDetector:
    def __init__(self, static_mode=True, max_hands=1, min_conf=0.5):
        self.api = None
        try:
            import mediapipe as mp
            from mediapipe.tasks import python as mpp
            from mediapipe.tasks.python import vision as mpv

            p = ensure_model_file()
            opts = mpv.HandLandmarkerOptions(
                base_options=mpp.BaseOptions(model_asset_path=p),
                num_hands=max_hands,
                min_hand_detection_confidence=min_conf,
                min_hand_presence_confidence=min_conf,
                min_tracking_confidence=0.5,
            )
            self.detector = mpv.HandLandmarker.create_from_options(opts)
            self.mp = mp
            self.api = "tasks"
            return
        except Exception as e:
            log_warn(f"MediaPipe tasks API failed: {e}")
        try:
            import mediapipe as mp

            self.detector = mp.solutions.hands.Hands(
                static_image_mode=static_mode,
                max_num_hands=max_hands,
                min_detection_confidence=min_conf,
                min_tracking_confidence=0.5,
            )
            self.mp = mp
            self.api = "solutions"
            return
        except Exception as e:
            log_err(f"MediaPipe init failed: {e}")
            raise RuntimeError(f"MediaPipe init failed: {e}")

    def detect(self, frame_bgr):
        rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        if self.api == "tasks":
            img = self.mp.Image(image_format=self.mp.ImageFormat.SRGB, data=rgb)
            res = self.detector.detect(img)
            if res.hand_landmarks and len(res.hand_landmarks) > 0:
                c = []
                for lm in res.hand_landmarks[0]:
                    c.extend([lm.x, lm.y, lm.z])
                return np.array(c, dtype=np.float32)
        else:
            res = self.detector.process(rgb)
            if res.multi_hand_landmarks:
                c = []
                for lm in res.multi_hand_landmarks[0].landmark:
                    c.extend([lm.x, lm.y, lm.z])
                return np.array(c, dtype=np.float32)
        return None

    def close(self):
        if hasattr(self.detector, "close"):
            self.detector.close()

## 5. Data Preprocessing Functions

In [9]:
def to_scalar(v, default=None):
    if v is None:
        return default
    if isinstance(v, np.ndarray):
        if v.shape == ():
            v = v.item()
        elif v.size == 1:
            v = v.reshape(()).item()
    if isinstance(v, bytes):
        return v.decode("utf-8")
    return v


def resample(seq, target=SEQ_LEN):
    """Resample sequence to target length using linear interpolation."""
    seq = np.asarray(seq, dtype=np.float32)
    if seq.ndim != 2:
        return np.zeros((target, RAW_DIM), dtype=np.float32)
    n, d = seq.shape
    if n == 0:
        return np.zeros((target, d), dtype=np.float32)
    if n == target:
        return seq.copy()
    x_old = np.linspace(0.0, 1.0, n, dtype=np.float32)
    x_new = np.linspace(0.0, 1.0, target, dtype=np.float32)
    out = np.zeros((target, d), dtype=np.float32)
    for i in range(d):
        out[:, i] = np.interp(x_new, x_old, seq[:, i]).astype(np.float32)
    return out


def to_raw_sequence(seq, target_len=None):
    """Convert various formats to raw sequence (T, RAW_DIM)."""
    try:
        arr = np.asarray(seq, dtype=np.float32)
    except Exception:
        return None
    if arr.size == 0:
        t = 0 if target_len is None else target_len
        return np.zeros((t, RAW_DIM), dtype=np.float32)
    if arr.ndim == 3 and arr.shape[1] == NUM_LANDMARKS and arr.shape[2] == NUM_COORDS:
        arr = arr.reshape(arr.shape[0], RAW_DIM)
    elif arr.ndim == 2 and arr.shape == (NUM_LANDMARKS, NUM_COORDS):
        arr = arr.reshape(1, RAW_DIM)
    elif arr.ndim == 2 and arr.shape[1] == RAW_DIM:
        pass
    elif arr.ndim == 2 and arr.shape[0] == RAW_DIM and arr.shape[1] != RAW_DIM:
        arr = arr.T
        if arr.shape[1] != RAW_DIM:
            return None
    elif arr.ndim == 1 and arr.size % RAW_DIM == 0:
        arr = arr.reshape(-1, RAW_DIM)
    else:
        return None
    arr = np.ascontiguousarray(arr, dtype=np.float32)
    if target_len is not None and arr.shape[0] != target_len:
        arr = resample(arr, target_len)
    return arr

In [10]:
def interp_extrap_1d(valid_idx, valid_vals, n):
    """Interpolate and extrapolate 1D values."""
    xi = np.arange(n, dtype=np.float32)
    xp = np.asarray(valid_idx, dtype=np.float32)
    fp = np.asarray(valid_vals, dtype=np.float32)
    yi = np.interp(xi, xp, fp).astype(np.float32)
    if len(valid_idx) >= 2:
        left = xi < valid_idx[0]
        if left.any():
            dx = float(valid_idx[1] - valid_idx[0])
            slope = 0.0 if dx == 0 else float((fp[1] - fp[0]) / dx)
            yi[left] = fp[0] + (xi[left] - valid_idx[0]) * slope
        right = xi > valid_idx[-1]
        if right.any():
            dx = float(valid_idx[-1] - valid_idx[-2])
            slope = 0.0 if dx == 0 else float((fp[-1] - fp[-2]) / dx)
            yi[right] = fp[-1] + (xi[right] - valid_idx[-1]) * slope
    return yi.astype(np.float32)


def interpolate_missing(lm_list):
    """Interpolate missing landmarks in a sequence."""
    n = len(lm_list)
    if n == 0:
        return np.zeros((0, RAW_DIM), dtype=np.float32)
    valid_idx = [i for i, lm in enumerate(lm_list) if lm is not None]
    if len(valid_idx) == 0:
        return np.zeros((n, RAW_DIM), dtype=np.float32)
    result = np.zeros((n, RAW_DIM), dtype=np.float32)
    valid_arr = []
    for i in valid_idx:
        lm = np.asarray(lm_list[i], dtype=np.float32).reshape(-1)
        if lm.size < RAW_DIM:
            pad = np.zeros((RAW_DIM,), dtype=np.float32)
            pad[: lm.size] = lm
            lm = pad
        elif lm.size > RAW_DIM:
            lm = lm[:RAW_DIM]
        valid_arr.append(lm.astype(np.float32))
        result[i] = lm.astype(np.float32)
    valid_arr = np.stack(valid_arr, axis=0)
    if len(valid_idx) == 1:
        result[:] = valid_arr[0]
        return result
    for d in range(RAW_DIM):
        result[:, d] = interp_extrap_1d(valid_idx, valid_arr[:, d], n)
    return result.astype(np.float32)

In [11]:
def compute_features(raw_seq):
    """Compute feature vector from raw landmark sequence.

    Features include:
    - Normalized landmarks (63 dims)
    - Velocity (63 dims)
    - Wrist velocity (3 dims)
    - Finger distances (10 dims)
    - Finger angles (5 dims)

    Total: 144 dims
    """
    raw_seq = to_raw_sequence(raw_seq)
    if raw_seq is None:
        shape = np.asarray(raw_seq).shape if raw_seq is not None else None
        raise ValueError(f"Invalid raw sequence shape: {shape}")
    T = raw_seq.shape[0]
    if T == 0:
        return np.zeros((0, FEATURE_DIM), dtype=np.float32)
    lms = raw_seq.reshape(T, NUM_LANDMARKS, NUM_COORDS)
    wrist = lms[:, WRIST_IDX, :]
    relative = lms - wrist[:, np.newaxis, :]
    mid = lms[:, MID_FINGER_IDX, :]
    palm_size = np.maximum(np.linalg.norm(mid - wrist, axis=-1, keepdims=True), 1e-6)
    norm_lms = relative / palm_size[:, np.newaxis, :]
    norm_flat = norm_lms.reshape(T, -1).astype(np.float32)
    vel = np.zeros_like(norm_flat)
    if T > 1:
        vel[1:] = norm_flat[1:] - norm_flat[:-1]
        vel[0] = vel[1]
    wrist_vel = np.zeros((T, 3), dtype=np.float32)
    if T > 1:
        wrist_vel[1:] = wrist[1:] - wrist[:-1]
        wrist_vel[0] = wrist_vel[1]
    dists = np.zeros((T, N_PAIRS), dtype=np.float32)
    for k, (i, j) in enumerate(PAIRS):
        dists[:, k] = np.linalg.norm(norm_lms[:, i] - norm_lms[:, j], axis=-1)
    angles = np.zeros((T, N_FINGERS), dtype=np.float32)
    for fi, chain in enumerate(FINGER_CHAINS):
        v1 = lms[:, chain[1]] - lms[:, chain[0]]
        v2 = lms[:, chain[-1]] - lms[:, chain[1]]
        n1 = np.linalg.norm(v1, axis=-1, keepdims=True) + 1e-8
        n2 = np.linalg.norm(v2, axis=-1, keepdims=True) + 1e-8
        cos_a = np.clip((v1 / n1 * v2 / n2).sum(-1), -1.0, 1.0)
        angles[:, fi] = np.arccos(cos_a)
    feat = np.concatenate([norm_flat, vel, wrist_vel, dists, angles], axis=1)
    return feat.astype(np.float32)


# Verify feature dimension
dummy = np.random.randn(SEQ_LEN, RAW_DIM).astype(np.float32)
f = compute_features(dummy)
print(f"Feature dim verified: {f.shape[1]} (expected {FEATURE_DIM})")

Feature dim verified: 144 (expected 144)


## 6. Data Augmentation Functions

In [12]:
def mirror_x(raw_seq):
    """Mirror landmarks along X axis."""
    raw = to_raw_sequence(raw_seq)
    if raw is None:
        raise ValueError("Invalid raw sequence for mirror_x")
    s = raw.copy().reshape(-1, NUM_LANDMARKS, NUM_COORDS)
    s[:, :, 0] = 1.0 - s[:, :, 0]
    return s.reshape(-1, RAW_DIM).astype(np.float32)


def rotate_2d(raw_seq, angle_deg):
    """Rotate landmarks around wrist."""
    raw = to_raw_sequence(raw_seq)
    if raw is None:
        raise ValueError("Invalid raw sequence for rotate_2d")
    s = raw.copy().reshape(-1, NUM_LANDMARKS, NUM_COORDS)
    wrist = s[:, WRIST_IDX : WRIST_IDX + 1, :2].copy()
    s[:, :, :2] -= wrist
    a = np.radians(angle_deg)
    c, sn = np.cos(a), np.sin(a)
    x = s[:, :, 0].copy()
    y = s[:, :, 1].copy()
    s[:, :, 0] = c * x - sn * y
    s[:, :, 1] = sn * x + c * y
    s[:, :, :2] += wrist
    return s.reshape(-1, RAW_DIM).astype(np.float32)


def scale_landmarks(raw_seq, factor):
    """Scale landmarks around wrist."""
    raw = to_raw_sequence(raw_seq)
    if raw is None:
        raise ValueError("Invalid raw sequence for scale_landmarks")
    s = raw.copy().reshape(-1, NUM_LANDMARKS, NUM_COORDS)
    wrist = s[:, WRIST_IDX : WRIST_IDX + 1, :].copy()
    s -= wrist
    s *= factor
    s += wrist
    return s.reshape(-1, RAW_DIM).astype(np.float32)


def add_jitter(raw_seq, sigma=0.003):
    """Add Gaussian noise to landmarks."""
    raw = to_raw_sequence(raw_seq)
    if raw is None:
        raise ValueError("Invalid raw sequence for add_jitter")
    noise = np.random.randn(*raw.shape).astype(np.float32) * sigma
    return (raw + noise).astype(np.float32)


def time_warp(raw_seq):
    """Apply time warping augmentation."""
    raw = to_raw_sequence(raw_seq)
    if raw is None:
        raise ValueError("Invalid raw sequence for time_warp")
    n = len(raw)
    if n < 4:
        return raw.copy()
    anchor = np.random.uniform(0.3, 0.7)
    warp = np.random.uniform(0.8, 1.2)
    x = np.linspace(0.0, 1.0, n, dtype=np.float32)
    x_new = np.where(
        x < anchor,
        x * warp,
        anchor * warp + (x - anchor) * (1.0 - anchor * warp) / (1.0 - anchor + 1e-8),
    )
    x_new = np.clip(x_new, 0.0, 1.0)
    out = np.zeros_like(raw)
    x_target = np.linspace(0.0, 1.0, n, dtype=np.float32)
    for d in range(raw.shape[1]):
        out[:, d] = np.interp(x_target, x_new, raw[:, d]).astype(np.float32)
    return out.astype(np.float32)


def speed_change(raw_seq):
    """Change speed of sequence."""
    raw = to_raw_sequence(raw_seq)
    if raw is None:
        raise ValueError("Invalid raw sequence for speed_change")
    n = len(raw)
    factor = np.random.uniform(0.8, 1.2)
    new_n = max(int(n * factor), SEQ_LEN // 2)
    x_old = np.linspace(0.0, 1.0, n, dtype=np.float32)
    x_new = np.linspace(0.0, 1.0, new_n, dtype=np.float32)
    out = np.zeros((new_n, raw.shape[1]), dtype=np.float32)
    for d in range(raw.shape[1]):
        out[:, d] = np.interp(x_new, x_old, raw[:, d]).astype(np.float32)
    return out.astype(np.float32)

## 7. Video Processing and Sample Generation

In [13]:
def extract_video(video_path, detector):
    """Extract hand landmarks from video."""
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return [], 0
    lms = []
    total = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        total += 1
        lms.append(detector.detect(frame))
    cap.release()
    return lms, total


def make_samples(lm_list, total_frames, class_name, is_train):
    """Generate training samples from landmark list."""
    valid = sum(1 for lm in lm_list if lm is not None)
    if total_frames == 0 or valid / total_frames < MIN_VALID_RATIO:
        return [], []
    raw = interpolate_missing(lm_list)
    n = len(raw)
    label = CLASS_TO_IDX[class_name]
    samples = []
    labels = []

    # Base sample
    base = resample(raw, SEQ_LEN)
    samples.append(base.astype(np.float32))
    labels.append(label)

    if not is_train:
        return samples, labels

    # Time-based cropping for longer sequences
    if n > SEQ_LEN:
        for start_r in [0.0, 0.15, 0.25]:
            for end_r in [0.75, 0.85, 1.0]:
                s = int(n * start_r)
                e = int(n * end_r)
                if e - s >= SEQ_LEN // 2:
                    sub = resample(raw[s:e], SEQ_LEN)
                    samples.append(sub.astype(np.float32))
                    labels.append(label)

    # Jitter augmentation
    for _ in range(3):
        aug = add_jitter(raw.copy())
        aug = resample(aug, SEQ_LEN)
        samples.append(aug.astype(np.float32))
        labels.append(label)

    # Rotation augmentation
    for angle in [-15, -10, -5, 5, 10, 15]:
        rot = rotate_2d(raw.copy(), angle)
        rot = resample(rot, SEQ_LEN)
        samples.append(rot.astype(np.float32))
        labels.append(label)

    # Scale augmentation
    for sc in [0.85, 0.9, 1.1, 1.15]:
        scaled = scale_landmarks(raw.copy(), sc)
        scaled = resample(scaled, SEQ_LEN)
        samples.append(scaled.astype(np.float32))
        labels.append(label)

    # Time warp augmentation
    for _ in range(2):
        tw = time_warp(raw.copy())
        tw = resample(tw, SEQ_LEN)
        samples.append(tw.astype(np.float32))
        labels.append(label)

    # Speed change augmentation
    for _ in range(2):
        sp = speed_change(raw.copy())
        sp = resample(sp, SEQ_LEN)
        samples.append(sp.astype(np.float32))
        labels.append(label)

    # Mirror augmentation (for all classes, no label swap)
    mir = mirror_x(raw.copy())
    mir = resample(mir, SEQ_LEN)
    samples.append(mir.astype(np.float32))
    labels.append(label)

    # Reverse time for non-swipe gestures
    if class_name not in SWIPE_CLASSES:
        rev = raw[::-1].copy()
        rev = resample(rev, SEQ_LEN)
        samples.append(rev.astype(np.float32))
        labels.append(label)

    return samples, labels

## 8. Dataset Loading

In [14]:
def save_cache(cache_path, samples, labels, is_train):
    try:
        cache_path = Path(cache_path)
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        if len(samples) == 0:
            sample_arr = np.zeros((0, SEQ_LEN, RAW_DIM), dtype=np.float32)
        else:
            sample_arr = np.stack(
                [to_raw_sequence(s, target_len=SEQ_LEN) for s in samples], axis=0
            ).astype(np.float32)
        label_arr = np.asarray(labels, dtype=np.int64)
        np.savez_compressed(
            str(cache_path),
            cache_version=np.array(CACHE_VERSION),
            sample_format=np.array("raw_landmarks"),
            raw_dim=np.array(RAW_DIM),
            seq_len=np.array(SEQ_LEN),
            is_train=np.array(int(is_train)),
            samples=sample_arr,
            labels=label_arr,
        )
        log_info(f"Cache saved: {cache_path}")
    except Exception as e:
        log_warn(f"Failed to save cache {cache_path}: {e}")


def try_load_cache(cache_path, is_train):
    cache_path = Path(cache_path)
    if not cache_path.exists():
        return None
    log_info(f"Loading cache: {cache_path}")
    try:
        with np.load(str(cache_path), allow_pickle=True) as c:
            version = (
                to_scalar(c["cache_version"]) if "cache_version" in c.files else None
            )
            sample_format = (
                to_scalar(c["sample_format"]) if "sample_format" in c.files else None
            )
            raw_dim = to_scalar(c["raw_dim"]) if "raw_dim" in c.files else None
            seq_len = to_scalar(c["seq_len"]) if "seq_len" in c.files else None
            cache_is_train = to_scalar(c["is_train"]) if "is_train" in c.files else None
            if (
                version != CACHE_VERSION
                or sample_format != "raw_landmarks"
                or raw_dim != RAW_DIM
                or seq_len != SEQ_LEN
                or cache_is_train != int(is_train)
            ):
                raise ValueError("cache metadata mismatch")
            samples = np.asarray(c["samples"], dtype=np.float32)
            labels = np.asarray(c["labels"], dtype=np.int64)
        if (
            samples.ndim != 3
            or samples.shape[1] != SEQ_LEN
            or samples.shape[2] != RAW_DIM
        ):
            raise ValueError(f"cache sample shape mismatch: {samples.shape}")
        if len(samples) != len(labels):
            raise ValueError("cache sample/label count mismatch")
        out_samples = [np.ascontiguousarray(s, dtype=np.float32) for s in samples]
        out_labels = labels.astype(np.int64).tolist()
        log_info(f"  {len(out_samples)} samples loaded")
        return out_samples, out_labels
    except Exception as e:
        log_warn(f"Cache incompatible, rebuilding from videos: {e}")
        return None


def sanitize_dataset(samples, labels, name):
    fixed_samples = []
    fixed_labels = []
    if len(samples) != len(labels):
        log_warn(f"{name}: sample/label count mismatch, using min length")
    for s, l in zip(samples, labels):
        raw = to_raw_sequence(s, target_len=SEQ_LEN)
        if raw is None:
            continue
        fixed_samples.append(raw.astype(np.float32))
        fixed_labels.append(int(l))
    skipped = len(list(zip(samples, labels))) - len(fixed_samples)
    if skipped > 0:
        log_warn(f"{name}: skipped {skipped} invalid samples")
    return fixed_samples, fixed_labels

In [15]:
def load_dataset(data_dir, detector, cache_path=None, is_train=True):
    cached = try_load_cache(cache_path, is_train)
    if cached is not None:
        return cached

    all_samples, all_labels = [], []
    counts = defaultdict(int)
    dp = Path(data_dir)
    for cn in CLASS_NAMES:
        cd = dp / cn
        if not cd.exists():
            log_warn(f"Not found: {cd}")
            continue
        vfs = sorted(
            f
            for f in cd.iterdir()
            if f.suffix.lower() in (".mp4", ".avi", ".mov", ".mkv", ".webm")
        )
        log_info(f"  [{cn}]: {len(vfs)} videos")
        for vf in tqdm(vfs, desc=f"  {cn}", leave=False):
            lm_list, total = extract_video(vf, detector)
            if total == 0:
                continue
            samps, labs = make_samples(lm_list, total, cn, is_train)
            for s, l in zip(samps, labs):
                raw = to_raw_sequence(s, target_len=SEQ_LEN)
                if raw is None:
                    continue
                all_samples.append(raw.astype(np.float32))
                all_labels.append(int(l))
                counts[CLASS_NAMES[l]] += 1

    log_info("Dataset statistics:")
    for cn in CLASS_NAMES:
        log_info(f"  {cn:>15s}: {counts[cn]:>5d} samples")
    log_info(f"  {'Total':>15s}: {len(all_samples):>5d} samples")

    if cache_path:
        save_cache(cache_path, all_samples, all_labels, is_train)

    return all_samples, all_labels

In [16]:
# Debug: Check video files and hand detection
detector = HandDetector(static_mode=True, max_hands=1, min_conf=0.5)

# Check what files exist in the dataset
print("=== Checking dataset structure ===")
for split in ["Train", "Test"]:
    split_dir = DATASET_PATH / split
    if split_dir.exists():
        print(f"\n{split}:")
        for cn in CLASS_NAMES:
            cd = split_dir / cn
            if cd.exists():
                files = list(cd.iterdir())
                video_files = [
                    f for f in files
                    if f.suffix.lower() in (".mp4", ".avi", ".mov", ".mkv", ".webm")
                ]
                print(f"  {cn}: {len(video_files)} videos")
    else:
        print(f"\n{split}: directory not found")

# Test hand detection on a sample video
print("\n=== Testing hand detection ===")
sample_video = None
for cn in CLASS_NAMES:
    cd = TRAIN_DIR / cn
    if cd.exists():
        for f in cd.iterdir():
            if f.suffix.lower() in (".mp4", ".avi", ".mov", ".mkv", ".webm"):
                sample_video = f
                break
    if sample_video:
        break

if sample_video:
    print(f"Testing on: {sample_video}")
    cap = cv2.VideoCapture(str(sample_video))
    print(f"  Video opened: {cap.isOpened()}")
    print(f"  Frame count: {int(cap.get(cv2.CAP_PROP_FRAME_COUNT))}")
    print(f"  FPS: {cap.get(cv2.CAP_PROP_FPS)}")

    # Test detection on first 10 frames
    detected = 0
    total_test = 0
    for i in range(10):
        ret, frame = cap.read()
        if not ret:
            break
        total_test += 1
        lm = detector.detect(frame)
        if lm is not None:
            detected += 1
    cap.release()
    print(f"  Hand detected in {detected}/{total_test} frames")
else:
    print("No video files found in expected formats!")

detector.close()

=== Checking dataset structure ===

Train:
  swipe_up: 24 videos
  swipe_down: 24 videos
  grab: 24 videos
  release: 16 videos
  noise: 115 videos

Test:
  swipe_up: 6 videos
  swipe_down: 6 videos
  grab: 6 videos
  release: 4 videos
  noise: 29 videos

=== Testing hand detection ===
Testing on: D:\Scarleto\Documents\Projet\comp5201\Projet\AirGesture-xuranus\dataset\processed\Train\swipe_up\swipe_up_001.mp4
  Video opened: True
  Frame count: 46
  FPS: 29.0


d:\Miniconda\envs\comp5201\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


  Hand detected in 10/10 frames


In [17]:
# Load datasets
detector = HandDetector(static_mode=True, max_hands=1, min_conf=0.5)

# Use local paths
cache_dir = NOTEBOOK_DIR / "checkpoints" / "cache"
cache_dir.mkdir(parents=True, exist_ok=True)
tr_cache = cache_dir / f"train_{CACHE_VERSION}.npz"
te_cache = cache_dir / f"test_{CACHE_VERSION}.npz"

log_info("[Phase 1] Extracting landmarks ...")
train_samples, train_labels = load_dataset(TRAIN_DIR, detector, tr_cache, is_train=True)
test_samples, test_labels = load_dataset(TEST_DIR, detector, te_cache, is_train=False)
detector.close()

train_samples, train_labels = sanitize_dataset(train_samples, train_labels, "train")
test_samples, test_labels = sanitize_dataset(test_samples, test_labels, "test")

print(f"\nTrain samples: {len(train_samples)}")
print(f"Test samples: {len(test_samples)}")

[18:59:03] INFO - [Phase 1] Extracting landmarks ...
[18:59:03] INFO -   [swipe_up]: 24 videos


[18:59:35] INFO -   [swipe_down]: 24 videos


[19:00:04] INFO -   [grab]: 24 videos


[19:00:44] INFO -   [release]: 16 videos


[19:01:07] INFO -   [noise]: 115 videos


...

[19:04:03] INFO -   [swipe_down]: 6 videos


[19:04:11] INFO -   [grab]: 6 videos


[19:04:20] INFO -   [release]: 4 videos


[19:04:25] INFO -   [noise]: 29 videos


[19:05:06] INFO - Dataset statistics:
[19:05:06] INFO -          swipe_up:     5 samples
[19:05:06] INFO -        swipe_down:     5 samples
[19:05:06] INFO -              grab:     6 samples
[19:05:06] INFO -           release:     4 samples
[19:05:06] INFO -             noise:    22 samples
[19:05:06] INFO -             Total:    42 samples
[19:05:06] INFO - Cache saved: d:\Scarleto\Documents\Projet\comp5201\Projet\AirGesture-xuranus\modelTraining\checkpoints\cache\test_v7_5class.npz

Train samples: 4769
Test samples: 42


## 9. Normalization Statistics

In [18]:
def compute_norm_stats(samples):
    sum_feat = None
    sum_sq_feat = None
    total_frames = 0
    bad = 0

    for s in tqdm(samples, desc="Computing stats", leave=False):
        raw = to_raw_sequence(s, target_len=SEQ_LEN)
        if raw is None:
            bad += 1
            continue
        feat = compute_features(raw).astype(np.float64)
        cur_sum = feat.sum(axis=0)
        cur_sq = np.square(feat).sum(axis=0)
        if sum_feat is None:
            sum_feat = cur_sum
            sum_sq_feat = cur_sq
        else:
            sum_feat += cur_sum
            sum_sq_feat += cur_sq
        total_frames += feat.shape[0]

    if total_frames == 0:
        raise RuntimeError("No valid samples for normalization")

    mean64 = sum_feat / total_frames
    var64 = np.maximum(sum_sq_feat / total_frames - np.square(mean64), 1e-12)
    mean = mean64.astype(np.float32)
    std = np.sqrt(var64).astype(np.float32)

    if bad > 0:
        log_warn(f"Skipped {bad} invalid samples while computing normalization stats")

    return {"mean": mean, "std": std}


log_info("[Phase 2] Computing normalization stats ...")
norm_stats = compute_norm_stats(train_samples)
print(f"Mean range: [{norm_stats['mean'].min():.4f}, {norm_stats['mean'].max():.4f}]")
print(f"Std range:  [{norm_stats['std'].min():.4f}, {norm_stats['std'].max():.4f}]")

[19:05:12] INFO - [Phase 2] Computing normalization stats ...


Mean range: [-0.9381, 1.6489]
Std range:  [0.0000, 1.1388]


## 10. PyTorch Dataset and DataLoader

In [19]:
class GestureDataset(Dataset):
    def __init__(self, raw_samples, labels, norm_stats=None, augment=False):
        self.raw_samples = raw_samples
        self.labels = labels
        self.norm_stats = norm_stats
        self.augment = augment

    def __len__(self):
        return len(self.raw_samples)

    def __getitem__(self, idx):
        raw = self.raw_samples[idx].copy()
        label = self.labels[idx]
        if self.augment and random.random() < 0.5:
            raw = add_jitter(raw, sigma=0.002)
        if self.augment and random.random() < 0.3:
            raw = time_warp(raw)
            raw = resample(raw, SEQ_LEN)
        feat = compute_features(raw)
        if self.norm_stats is not None:
            feat = (feat - self.norm_stats["mean"]) / (self.norm_stats["std"] + 1e-8)
        x = torch.FloatTensor(feat.T)
        y = torch.tensor(label, dtype=torch.long)
        return x, y


def compute_class_weights(labels):
    counts = np.bincount(labels, minlength=NUM_CLASSES).astype(float)
    counts = np.maximum(counts, 1.0)
    w = counts.sum() / (NUM_CLASSES * counts)
    w = w / w.sum() * NUM_CLASSES
    return torch.FloatTensor(w)


def make_sampler(labels):
    counts = np.bincount(labels, minlength=NUM_CLASSES).astype(float)
    counts = np.maximum(counts, 1.0)
    sw = [1.0 / counts[l] for l in labels]
    return WeightedRandomSampler(sw, len(sw), replacement=True)

In [20]:
# Create dataloaders
class_weights = compute_class_weights(train_labels)
print(f"Class weights: {class_weights.numpy().round(3)}")

sampler = make_sampler(train_labels)
train_dataset = GestureDataset(train_samples, train_labels, norm_stats, augment=True)
test_dataset = GestureDataset(test_samples, test_labels, norm_stats, augment=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

Class weights: [1.328 1.055 1.019 1.364 0.234]
Train batches: 150
Test batches: 2


## 11. Model Definition (GestureTCN)

In [21]:
class CausalConv1d(nn.Module):
    def __init__(self, in_ch, out_ch, ks, dilation=1):
        super().__init__()
        self.pad = (ks - 1) * dilation
        self.conv = nn.Conv1d(
            in_ch, out_ch, ks, padding=self.pad, dilation=dilation, bias=False
        )

    def forward(self, x):
        o = self.conv(x)
        if self.pad > 0:
            o = o[:, :, : -self.pad]
        return o


class ResBlock(nn.Module):
    def __init__(self, ch, ks=3, dilation=1, dropout=0.15):
        super().__init__()
        self.net = nn.Sequential(
            CausalConv1d(ch, ch, ks, dilation),
            nn.BatchNorm1d(ch),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            CausalConv1d(ch, ch, ks, dilation),
            nn.BatchNorm1d(ch),
        )
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.act(self.net(x) + x)


class ChannelBlock(nn.Module):
    def __init__(self, in_ch, out_ch, ks=3, dilation=1, dropout=0.15):
        super().__init__()
        self.net = nn.Sequential(
            CausalConv1d(in_ch, out_ch, ks, dilation),
            nn.BatchNorm1d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            CausalConv1d(out_ch, out_ch, ks, dilation),
            nn.BatchNorm1d(out_ch),
        )
        self.skip = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm1d(out_ch),
        )
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.act(self.net(x) + self.skip(x))


class GestureTCN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, feat_dim=FEATURE_DIM, dropout=0.15):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(feat_dim, 48, 1, bias=False),
            nn.BatchNorm1d(48),
            nn.ReLU(inplace=True),
        )
        self.blocks = nn.Sequential(
            ResBlock(48, 3, 1, dropout),
            ResBlock(48, 3, 2, dropout),
            ChannelBlock(48, 64, 3, 4, dropout),
            ResBlock(64, 3, 1, dropout),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(32, num_classes),
        )
        self._init()

    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.pool(x).squeeze(-1)
        return self.head(x)


# Create model
model = GestureTCN(NUM_CLASSES, FEATURE_DIM, dropout=0.15).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model params: {total_params:,}")

Model params: 87,077


## 12. Training Loop

In [22]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    if loader is None:
        return 0.0, 0.0, [], []
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    preds_all, labels_all = [], []
    for bx, by in loader:
        bx, by = bx.to(device), by.to(device)
        logits = model(bx)
        loss = criterion(logits, by)
        total_loss += loss.item() * bx.size(0)
        p = logits.argmax(1)
        correct += (p == by).sum().item()
        total += bx.size(0)
        preds_all.extend(p.cpu().numpy())
        labels_all.extend(by.cpu().numpy())
    if total == 0:
        return 0.0, 0.0, preds_all, labels_all
    return total_loss / total, correct / total, preds_all, labels_all

In [23]:
# Training setup
save_dir = NOTEBOOK_DIR / "checkpoints"
save_dir.mkdir(parents=True, exist_ok=True)
ckpt_path = save_dir / "gesture_tcn_best.pth"

try:
    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(DEVICE), label_smoothing=0.1
    )
except TypeError:
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

best_acc = -1.0
best_epoch = 0
patience_ctr = 0

In [24]:
# Training loop
log_info("[Phase 4] Training ...")

for epoch in range(1, EPOCHS + 1):
    model.train()
    t_loss, t_correct, t_total = 0.0, 0, 0
    for bx, by in train_loader:
        bx, by = bx.to(DEVICE), by.to(DEVICE)
        optimizer.zero_grad()
        logits = model(bx)
        loss = criterion(logits, by)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        optimizer.step()
        t_loss += loss.item() * bx.size(0)
        t_correct += (logits.argmax(1) == by).sum().item()
        t_total += bx.size(0)

    scheduler.step()

    tr_loss = t_loss / max(t_total, 1)
    tr_acc = t_correct / max(t_total, 1)
    te_loss, te_acc, _, _ = evaluate(model, test_loader, criterion, DEVICE)
    lr_now = optimizer.param_groups[0]["lr"]

    tag = ""
    if te_acc > best_acc:
        best_acc = te_acc
        best_epoch = epoch
        patience_ctr = 0
        torch.save(model.state_dict(), ckpt_path)
        tag = "  <- best"
    else:
        patience_ctr += 1

    if epoch % 5 == 0 or tag:
        log_info(
            f"Epoch {epoch:>3d}/{EPOCHS} | "
            f"TrL:{tr_loss:.4f} TrA:{tr_acc:.4f} | "
            f"TeL:{te_loss:.4f} TeA:{te_acc:.4f} | "
            f"LR:{lr_now:.2e}{tag}"
        )

    if patience_ctr >= PATIENCE:
        log_info(f"Early stop at epoch {epoch}")
        break

log_info(f"Best test acc: {max(best_acc, 0.0):.4f} @ epoch {best_epoch}")

[19:05:31] INFO - [Phase 4] Training ...
[19:05:34] INFO - Epoch   1/300 | TrL:0.6810 TrA:0.7754 | TeL:1.0209 TeA:0.7381 | LR:2.00e-03  <- best
[19:05:36] INFO - Epoch   2/300 | TrL:0.4614 TrA:0.9488 | TeL:0.9238 TeA:0.8571 | LR:2.00e-03  <- best
[19:05:39] INFO - Epoch   4/300 | TrL:0.4185 TrA:0.9824 | TeL:0.8719 TeA:0.8810 | LR:2.00e-03  <- best
[19:05:41] INFO - Epoch   5/300 | TrL:0.4110 TrA:0.9897 | TeL:0.8092 TeA:0.9048 | LR:2.00e-03  <- best
[19:05:50] INFO - Epoch  10/300 | TrL:0.4016 TrA:0.9920 | TeL:0.8235 TeA:0.9048 | LR:1.99e-03
[19:05:52] INFO - Epoch  11/300 | TrL:0.3975 TrA:0.9939 | TeL:0.8253 TeA:0.9286 | LR:1.99e-03  <- best
[19:06:00] INFO - Epoch  15/300 | TrL:0.3955 TrA:0.9945 | TeL:0.8244 TeA:0.9286 | LR:1.99e-03
[19:06:10] INFO - Epoch  20/300 | TrL:0.3913 TrA:0.9929 | TeL:0.7901 TeA:0.9524 | LR:1.98e-03  <- best
[19:06:20] INFO - Epoch  25/300 | TrL:0.3936 TrA:0.9962 | TeL:0.8788 TeA:0.8810 | LR:1.97e-03
[19:06:30] INFO - Epoch  30/300 | TrL:0.3879 TrA:0.9981 | T

In [25]:
# Load best model
model.load_state_dict(torch.load(str(ckpt_path), map_location=DEVICE, weights_only=True))

<All keys matched successfully>

## 13. Evaluation

In [26]:
log_info("[Phase 5] Evaluation ...")
crit = nn.CrossEntropyLoss()
_, te_acc, preds, gts = evaluate(model, test_loader, crit, DEVICE)
log_info(f"Accuracy: {te_acc:.4f}")

label_ids = list(range(NUM_CLASSES))
if len(gts) > 0:
    print("\nClassification Report:")
    print(
        classification_report(
            gts,
            preds,
            labels=label_ids,
            target_names=CLASS_NAMES,
            digits=4,
            zero_division=0,
        )
    )

    cm = confusion_matrix(gts, preds, labels=label_ids)
    print("\nConfusion Matrix:")
    hdr = "          " + "  ".join(f"{n[:6]:>6s}" for n in CLASS_NAMES)
    print(hdr)
    for i, row in enumerate(cm):
        print(f"  {CLASS_NAMES[i][:8]:>8s}  " + "  ".join(f"{v:>6d}" for v in row))

[19:08:08] INFO - [Phase 5] Evaluation ...
[19:08:08] INFO - Accuracy: 0.9524

Classification Report:
              precision    recall  f1-score   support

    swipe_up     1.0000    0.8000    0.8889         5
  swipe_down     1.0000    1.0000    1.0000         5
        grab     1.0000    0.8333    0.9091         6
     release     1.0000    1.0000    1.0000         4
       noise     0.9167    1.0000    0.9565        22

    accuracy                         0.9524        42
   macro avg     0.9833    0.9267    0.9509        42
weighted avg     0.9563    0.9524    0.9510        42


Confusion Matrix:
          swipe_  swipe_    grab  releas   noise
  swipe_up       4       0       0       0       1
  swipe_do       0       5       0       0       0
      grab       0       0       5       0       1
   release       0       0       0       4       0
     noise       0       0       0       0      22


## 14. Pruning

In [27]:
def apply_pruning(model, amount):
    log_info(f"Pruning {amount:.0%} ...")
    for m in model.modules():
        if isinstance(m, (nn.Linear, nn.Conv1d)):
            prune.l1_unstructured(m, "weight", amount=amount)
    return model


def remove_pruning(model):
    for m in model.modules():
        if isinstance(m, (nn.Linear, nn.Conv1d)):
            try:
                prune.remove(m, "weight")
            except ValueError:
                pass


def sparsity_report(model):
    tp, tz = 0, 0
    for name, p in model.named_parameters():
        if "weight" in name:
            n = p.numel()
            z = (p == 0).sum().item()
            tp += n
            tz += z
            log_info(f"  {name:>45s} | {n:>7d} | {z / max(n, 1):>5.1%}")
    log_info(f"  {'Total':>45s} | {tp:>7d} | {tz / max(tp, 1):>5.1%}")


log_info("[Phase 6] Pruning ...")
model = apply_pruning(model, PRUNE_AMOUNT)
_, pa, _, _ = evaluate(model, test_loader, crit, DEVICE)
log_info(f"Post-pruning acc: {pa:.4f}")
remove_pruning(model)
sparsity_report(model)

[19:08:42] INFO - [Phase 6] Pruning ...
[19:08:42] INFO - Pruning 20% ...
[19:08:42] INFO - Post-pruning acc: 0.9524
[19:08:42] INFO -                                   stem.0.weight |    6912 | 20.0%
[19:08:42] INFO -                                   stem.1.weight |      48 |  0.0%
[19:08:42] INFO -                      blocks.0.net.0.conv.weight |    6912 | 20.0%
[19:08:42] INFO -                           blocks.0.net.1.weight |      48 |  0.0%
[19:08:42] INFO -                      blocks.0.net.4.conv.weight |    6912 | 20.0%
[19:08:42] INFO -                           blocks.0.net.5.weight |      48 |  0.0%
[19:08:42] INFO -                      blocks.1.net.0.conv.weight |    6912 | 20.0%
[19:08:42] INFO -                           blocks.1.net.1.weight |      48 |  0.0%
[19:08:42] INFO -                      blocks.1.net.4.conv.weight |    6912 | 20.0%
[19:08:42] INFO -                           blocks.1.net.5.weight |      48 |  0.0%
[19:08:42] INFO -                      bloc

## 15. Export Models

In [ ]:
# Install onnxscript (uncomment if needed)
# !pip install onnxscript

In [28]:
def export_torchscript(model, save_dir):
    m = deepcopy(model).cpu().eval()
    d = torch.randn(1, FEATURE_DIM, SEQ_LEN)
    p = Path(save_dir) / "gesture_tcn.pt"
    try:
        t = torch.jit.trace(m, d)
        t.save(str(p))
        log_info(f"TorchScript: {p}")
    except Exception as e:
        log_warn(f"TorchScript failed: {e}")


def export_onnx(model, save_dir):
    m = deepcopy(model).cpu().eval()
    d = torch.randn(1, FEATURE_DIM, SEQ_LEN)
    p = Path(save_dir) / "gesture_tcn.onnx"
    try:
        torch.onnx.export(
            m,
            d,
            str(p),
            input_names=["input"],
            output_names=["logits"],
            dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
            opset_version=14,
        )
        log_info(f"ONNX: {p}")
    except Exception as e:
        log_warn(f"ONNX failed: {e}")


def export_int8(model, save_dir):
    try:
        from torch.quantization import quantize_dynamic

        m = deepcopy(model).cpu().eval()
        q = quantize_dynamic(m, {nn.Linear}, dtype=torch.qint8)
        p = Path(save_dir) / "gesture_tcn_int8.pt"
        torch.save(q.state_dict(), str(p))
        log_info(f"INT8: {p}")
    except Exception as e:
        log_warn(f"INT8 failed: {e}")

In [33]:
def save_all(model, norm_stats, save_dir):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    
    pp = save_dir / "gesture_tcn_pruned.pth"
    torch.save(model.state_dict(), str(pp))
    log_info(f"Weights: {pp}")
    
    cfg = {
        "cache_version": CACHE_VERSION,
        "class_names": CLASS_NAMES,
        "seq_len": SEQ_LEN,
        "feature_dim": FEATURE_DIM,
        "raw_dim": RAW_DIM,
        "num_classes": NUM_CLASSES,
        "num_landmarks": NUM_LANDMARKS,
        "normalize_mean": norm_stats["mean"].tolist(),
        "normalize_std": norm_stats["std"].tolist(),
        "pairs": PAIRS,
        "fingertip_ids": FINGERTIP_IDS,
        "base_ids": BASE_IDS,
        "n_fingers": N_FINGERS,
        "finger_chains": FINGER_CHAINS,
    }
    cp = save_dir / "config.json"
    with open(cp, "w", encoding="utf-8") as f:
        json.dump(cfg, f, indent=2, ensure_ascii=False)
    log_info(f"Config: {cp}")
    
    export_torchscript(model, save_dir)
    export_onnx(model, save_dir)
    export_int8(model, save_dir)


log_info("[Phase 7] Saving ...")
save_all(model, norm_stats, save_dir)

[19:12:57] INFO - [Phase 7] Saving ...
[19:12:57] INFO - Weights: d:\Scarleto\Documents\Projet\comp5201\Projet\AirGesture-xuranus\modelTraining\checkpoints\gesture_tcn_pruned.pth
[19:12:57] INFO - Config: d:\Scarleto\Documents\Projet\comp5201\Projet\AirGesture-xuranus\modelTraining\checkpoints\config.json
[19:12:57] INFO - TorchScript: d:\Scarleto\Documents\Projet\comp5201\Projet\AirGesture-xuranus\modelTraining\checkpoints\gesture_tcn.pt


C:\Users\Scarleto\AppData\Local\Temp\ipykernel_11320\940027835.py:18: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0321 19:13:02.039000 11320 site-packages\torch\onnx\_internal\exporter\_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 14 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `GestureTCN([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `GestureTCN([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


d:\Miniconda\envs\comp5201\Lib\copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...


The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 14).


[torch.onnx] Translate the graph into ONNX... ✅


Failed to convert the model to the target version 14 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "d:\Miniconda\envs\comp5201\Lib\site-packages\onnxscript\version_converter\__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Miniconda\envs\comp5201\Lib\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "d:\Miniconda\envs\comp5201\Lib\site-packages\onnxscript\version_converter\__init__.py", line 115, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Miniconda\envs\comp5201\Lib\site-packages\onnx\version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

Applied 20 of general pattern rewrite rules.
[19:13:17] INFO - ONNX: d:\Scarleto\Documents\Projet\comp5201\Projet\AirGesture-xuranus\modelTraining\checkpoints\gesture_tcn.onnx
[19:13:17] INFO - INT8: d:\Scarleto\Documents\Projet\comp5201\Projet\AirGesture-xuranus\modelTraining\checkpoints\gesture_tcn_int8.pt


C:\Users\Scarleto\AppData\Local\Temp\ipykernel_11320\940027835.py:37: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  q = quantize_dynamic(m, {nn.Linear}, dtype=torch.qint8)


## 16. Verify ONNX Model

In [34]:
# Verify ONNX model
import onnx
import onnxruntime as ort

onnx_path = save_dir / "gesture_tcn.onnx"

# Check ONNX model
onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)
print("ONNX model is valid!")

# Test inference
sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
dummy_input = np.random.randn(1, FEATURE_DIM, SEQ_LEN).astype(np.float32)
outputs = sess.run(None, {"input": dummy_input})
print(f"ONNX output shape: {outputs[0].shape}")
print(f"ONNX output (logits): {outputs[0]}")

ONNX model is valid!
ONNX output shape: (1, 5)
ONNX output (logits): [[-1.0673683  -0.36766273 -0.99028254 -1.0678798   0.48260978]]


## 17. Download Results

In [35]:
# List saved files
import os
print("Saved files:")
for f in sorted(save_dir.iterdir()):
    if f.is_file():
        size = f.stat().st_size
        print(f"  {f.name}: {size:,} bytes")

Saved files:
  config.json: 8,635 bytes
  gesture_tcn.onnx: 96,012 bytes
  gesture_tcn.onnx.data: 343,680 bytes
  gesture_tcn.pt: 478,169 bytes
  gesture_tcn_best.pth: 375,833 bytes
  gesture_tcn_int8.pt: 369,841 bytes
  gesture_tcn_pruned.pth: 375,973 bytes


In [36]:
# Output paths (for reference)
print("Model files saved to:")
print(f"  ONNX: {save_dir / 'gesture_tcn.onnx'}")
print(f"  Config: {save_dir / 'config.json'}")
print(f"  PyTorch: {save_dir / 'gesture_tcn_pruned.pth'}")

Model files saved to:
  ONNX: d:\Scarleto\Documents\Projet\comp5201\Projet\AirGesture-xuranus\modelTraining\checkpoints\gesture_tcn.onnx
  Config: d:\Scarleto\Documents\Projet\comp5201\Projet\AirGesture-xuranus\modelTraining\checkpoints\config.json
  PyTorch: d:\Scarleto\Documents\Projet\comp5201\Projet\AirGesture-xuranus\modelTraining\checkpoints\gesture_tcn_pruned.pth


## Summary

This notebook has:
1. Loaded the gesture dataset from `../dataset/processed/`
2. Extracted hand landmarks using MediaPipe
3. Computed features (normalized landmarks, velocity, distances, angles)
4. Trained a TCN model for gesture classification (5 classes)
5. Applied L1 pruning for model compression
6. Exported the model to ONNX format for Android deployment

**Classes (5)**:
- `swipe_up` - 上滑
- `swipe_down` - 下滑
- `grab` - 抓
- `release` - 放
- `noise` - 噪音

**Output files:**
- `gesture_tcn.onnx` - ONNX model for Android
- `gesture_tcn_pruned.pth` - PyTorch weights
- `gesture_tcn.pt` - TorchScript model
- `gesture_tcn_int8.pt` - INT8 quantized model
- `config.json` - Model configuration and normalization stats